In [ ]:
# Notebook: Bode Plot Construction for H(s) = (100s^2 + 750s) / (s^2 + 13s + 30)
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import matplotlib.pyplot as plt
import control as ct

# 1. Define Transfer Function H(s)
num_coeffs = [100.0, 750.0, 0.0]
den_coeffs = [1.0, 13.0, 30.0]
H = ct.tf(num_coeffs, den_coeffs)

# 2. Frequency range spanning from 0.01 to 1000 rad/s (logarithmic scale)
omega = np.logspace(-2, 3, 500) # rad/s

# --- Analytical Calculations based on Factored Form H(s) = 25 * [s(1 + s/75)] / [(1 + s/3)(1 + s/10)] ---
mag_const_db = 20 * np.log10(25.0) * np.ones_like(omega)
phase_const_deg = np.zeros_like(omega)

# Zero at origin (s = 0)
mag_zero0_db = 20 * np.log10(omega)
phase_zero0_deg = 90.0 * np.ones_like(omega)

# Simple zero at s = -7.5 (omega_z = 7.5)
omega_z = 7.5
mag_zero75_db = 20 * np.log10(np.sqrt(1.0 + (omega / omega_z)**2))
phase_zero75_deg = np.rad2deg(np.arctan(omega / omega_z))

# Simple pole at s = -3 (omega_1 = 3)
omega_p1 = 3.0
mag_pole3_db = -20 * np.log10(np.sqrt(1.0 + (omega / omega_p1)**2))
phase_pole3_deg = -np.rad2deg(np.arctan(omega / omega_p1))

# Simple pole at s = -10 (omega_2 = 10)
omega_p2 = 10.0
mag_pole10_db = -20 * np.log10(np.sqrt(1.0 + (omega / omega_p2)**2))
phase_pole10_deg = -np.rad2deg(np.arctan(omega / omega_p2))

# Total Exact Analytical Superposition
mag_total_exact_db = mag_const_db + mag_zero0_db + mag_zero75_db + mag_pole3_db + mag_pole10_db
phase_total_exact_deg = phase_const_deg + phase_zero0_deg + phase_zero75_deg + phase_pole3_deg + phase_pole10_deg

# 3. Using Python Control Library frequency response method (Warning-free & Unwrapped Phase)
response = H.frequency_response(omega)
mag_ct_db = 20 * np.log10(response.magnitude)
phase_ct_deg = np.rad2deg(np.unwrap(response.phase))

# 4. Plotting using Matplotlib (Subplots for Magnitude and Phase with external legends)
fig, (ax_mag, ax_phase) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# --- Magnitude Plot ---
ax_mag.semilogx(omega, mag_total_exact_db, 'b-', linewidth=2, label='Exact Curve (Analytical Superposition)')
ax_mag.semilogx(omega, mag_ct_db, 'g:', linewidth=2, label='Control Library Response')
ax_mag.axvline(omega_p1, color='gray', linestyle=':', alpha=0.7, label=f'Corner Frequency $\\omega_1 = {omega_p1}$ rad/s (Pole)')
ax_mag.axvline(omega_p2, color='orange', linestyle=':', alpha=0.7, label=f'Corner Frequency $\\omega_2 = {omega_p2}$ rad/s (Pole)')
ax_mag.axvline(omega_z, color='purple', linestyle=':', alpha=0.7, label=f'Corner Frequency $\\omega_3 = {omega_z}$ rad/s (Zero)')
ax_mag.set_ylabel('Magnitude [dB]', fontsize=11)
ax_mag.set_title(r'Bode Diagram of $\mathcal{H}(s) = \frac{100s^2+750s}{s^2+13s+30}$', fontsize=12)
ax_mag.grid(True, which="both", linestyle=":", alpha=0.6)
ax_mag.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9, frameon=True)

# --- Phase Plot ---
ax_phase.semilogx(omega, phase_total_exact_deg, 'b-', linewidth=2, label='Exact Curve (Analytical Superposition)')
ax_phase.semilogx(omega, phase_ct_deg, 'g:', linewidth=2, label='Control Library Response')
ax_phase.axvline(omega_p1, color='gray', linestyle=':', alpha=0.7)
ax_phase.axvline(omega_p2, color='orange', linestyle=':', alpha=0.7)
ax_phase.axvline(omega_z, color='purple', linestyle=':', alpha=0.7)
ax_phase.set_ylabel('Phase [degrees]', fontsize=11)
ax_phase.set_xlabel(r'Frequency $\omega$ [rad/s]', fontsize=11)
ax_phase.grid(True, which="both", linestyle=":", alpha=0.6)
ax_phase.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9, frameon=True)

plt.tight_layout()
plt.show()

# NOTE ON RIGOROUS ALIGNMENT:
# The complete overlay of both magnitude and phase curves stems from using the exact 
# factored DC gain (K = 25) derived in the analytical text. This guarantees that manual 
# superposition and computational system responses are fully consistent.